# Medical Guideline Extraction Pipeline – Google Colab (Full pipeline)

Run the **full pipeline** (Stages a→b→c→d→e) on Google Colab with GPU.

## Pipeline (5 stages)
| Stage | Input → Device → Output |
|-------|--------------------------|
| **a** | PDF_guidelines → extract_text → raw_text |
| **b** | raw_text → clean_segment → text_chunks |
| **c** | text_chunks → recognize_entities (NER + acronym expansion) → statements_with_medical_entities |
| **d** | statements_with_medical_entities → infer_entities (entity linking) → candidate_statements |
| **e** | candidate_statements → validate (LLM extraction) → validated_facts (subject, predicate, object, exception, duration) |

Stage e **extracts** factual statements (no factuality scoring); experts validate later.

## Setup
1. **Enable GPU**: Runtime → Change runtime type → **T4 GPU**
2. **Upload ZIP**: In Step 4, upload `Thesis_llama_colab.zip` (created by `create_colab_zip.py`)
3. **Optional**: Add PDFs under `data/` and/or `data/UMLS.csv` for entity linking
4. **Run all cells** in order

## Step 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install -q transformers accelerate torch sentencepiece
!pip install -q pymupdf pdfplumber
!pip install -q nltk pydantic huggingface_hub

# Download NLTK data (punkt_tab is required for recent NLTK versions)
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("[OK] All dependencies installed!")

## Step 2: Login to Hugging Face (Required for Llama)

You need a Hugging Face token to access Llama models.

**How to get your token:**
1. Go to: https://huggingface.co/settings/tokens
2. Create a new token (or copy existing one)
3. Accept Llama license: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
4. Paste your token below when prompted

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import getpass

# Option 1: Use Colab Secrets (recommended)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("[OK] Logged in using Colab Secrets")
except:
    # Option 2: Manual token input
    print("[*] Enter your Hugging Face token:")
    print("    Get it from: https://huggingface.co/settings/tokens")
    hf_token = getpass.getpass("Token: ")
    login(token=hf_token)
    print("[OK] Logged in successfully!")

print("\n[IMPORTANT] Make sure you've accepted the Llama license:")
print("            https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct")

## Step 3: Verify GPU Availability

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"[OK] GPU: {torch.cuda.get_device_name(0)}")
    print(f"[OK] GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"[OK] CUDA Version: {torch.version.cuda}")
    device = "cuda"
else:
    print("[!] No GPU available - will use CPU (slower)")
    device = "cpu"

## Step 4: Upload and Extract ZIP File

In [ ]:
from google.colab import files
import zipfile
import os

print("[*] Please upload your Thesis_llama_colab.zip file...")
uploaded = files.upload()

# Extract the ZIP file
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"\n[*] Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('/content')
        print(f"[OK] Extracted successfully!")
        
        # Find project root: either /content (if pipeline/ and data/ at top) or /content/Thesis_llama
        content_list = os.listdir('/content')
        if 'pipeline' in content_list and os.path.isdir('/content/pipeline'):
            project_dir = '/content'
        else:
            extracted_dirs = [d for d in content_list 
                             if os.path.isdir(f'/content/{d}') and ('Thesis' in d or 'pipeline' in os.listdir(f'/content/{d}'))]
            project_dir = f'/content/{extracted_dirs[0]}' if extracted_dirs else '/content'
        os.chdir(project_dir)
        print(f"[*] Project directory: {os.getcwd()}")
        print("\n[*] Project structure:")
        !ls -la

## Step 5: Verify Pipeline Import

In [ ]:
import sys
import os

# Project root: /content if pipeline/ is there, else first dir that contains pipeline
content_list = os.listdir('/content')
if 'pipeline' in content_list and os.path.isdir('/content/pipeline'):
    project_path = '/content'
else:
    subdirs = [d for d in content_list if os.path.isdir(f'/content/{d}')]
    project_path = None
    for d in subdirs:
        if os.path.isdir(f'/content/{d}/pipeline'):
            project_path = f'/content/{d}'
            break
    project_path = project_path or '/content'
sys.path.insert(0, project_path)
os.chdir(project_path)
print(f"[OK] Project path: {project_path}")
print(f"[OK] Working directory: {os.getcwd()}")

try:
    from pipeline.pipeline import Pipeline
    print("[OK] Pipeline imported successfully!")
except Exception as e:
    print(f"[ERROR] Import error: {e}")
    !ls -la

## Step 6: Verify UMLS Database

The pipeline now supports UMLS entity linking for better concept normalization.
If `data/UMLS.csv` exists in your ZIP, it will be used automatically.
Otherwise, the pipeline will use rule-based linking (fallback).

In [ ]:
import os
from pathlib import Path

# Check for UMLS database
umls_path = Path("data/UMLS.csv")
if umls_path.exists():
    umls_size_mb = umls_path.stat().st_size / (1024 * 1024)
    print(f"[OK] UMLS database found: {umls_path}")
    print(f"     Size: {umls_size_mb:.1f} MB")
    print(f"     Will be used for entity linking")
else:
    print(f"[!] UMLS database not found: {umls_path}")
    print(f"     Pipeline will use rule-based linking (fallback)")
    print(f"     To use UMLS: upload UMLS.csv to data/ folder")

# Check for PDFs
os.makedirs('data', exist_ok=True)
pdf_files = [f for f in os.listdir('data') if f.endswith('.pdf')]
print(f"\n[*] PDFs in data/: {len(pdf_files)}")
if pdf_files:
    print(f"    Files: {pdf_files[:5]}{'...' if len(pdf_files) > 5 else ''}")
else:
    print(f"    [!] No PDFs found - upload PDFs to data/ folder")
    print(f"    [*] You can upload PDFs manually or they should be in your ZIP file")

## Step 7: Run full pipeline (Stages a→e)

Configure parameters and run. The pipeline will:
- **Stage a–b**: Extract text from PDFs in `data/`, chunk with parsing rules
- **Stage c**: NER + acronym expansion (same text for entity–SPO alignment)
- **Stage d**: Entity linking (UMLS if `data/UMLS.csv` present, else rule-based)
- **Stage e**: LLM **extraction** only: subject, predicate, object, exception, duration (chain-of-thought prompt). No factuality scoring—experts validate later.

In [ ]:
from pipeline.pipeline import Pipeline
import os
import sys
import time
from pathlib import Path

# Ensure we're in the project root
if not os.path.exists('pipeline'):
    content_list = os.listdir('/content')
    if 'pipeline' in content_list:
        os.chdir('/content')
    else:
        for d in content_list:
            if os.path.isdir(f'/content/{d}') and os.path.exists(f'/content/{d}/pipeline'):
                os.chdir(f'/content/{d}')
                break
    sys.path.insert(0, os.getcwd())

print(f"[*] Working directory: {os.getcwd()}")
print("[*] Running full pipeline (Stages a→b→c→d→e)...\n")

# Stage d: UMLS (optional)
umls_csv_path = "data/UMLS.csv"
use_umls = Path(umls_csv_path).exists()
if use_umls:
    print(f"[OK] UMLS found → entity linking (Stage d) will use it")
else:
    print(f"[!] No UMLS → rule-based linking (Stage d)")

# Stage c: acronyms (optional)
acronym_file = "pipeline/data/heart_failure_acronyms.json"
if not Path(acronym_file).exists():
    acronym_file = None

start_time = time.time()

# Parameters (tune as needed)
MIN_NER_SCORE = 0.55       # Stage c: 0.55 = more entities, 0.65 = stricter
MAX_EXTRACTION_TOKENS = 400  # Stage e: allows multiple statements per chunk
BATCH_SIZE = 6             # Stage e: 4–8 for T4 GPU

pipeline = Pipeline(
    pdf_dir="data",
    output_file="validated_output.json",
    min_chunk_chars=40,
    min_ner_score=MIN_NER_SCORE,
    max_validation_tokens=MAX_EXTRACTION_TOKENS,
    batch_size=BATCH_SIZE,
    umls_csv_path=umls_csv_path if use_umls else None,
    filter_unmatched_entities=use_umls,
    acronym_file=acronym_file,
    skip_llm_validation=False,  # Run Stage e (LLM extraction)
)

validated = pipeline.run()

elapsed = time.time() - start_time
print(f"\n{'='*70}")
print("[SUCCESS] Full pipeline complete!")
print(f"{'='*70}")
print(f"[*] Extracted statements: {validated.count()}")
print(f"[*] Time: {elapsed/60:.1f} min ({elapsed:.0f}s)")
print(f"[*] Output: validated_output.json (subject, predicate, object, exception, duration)")
print(f"{'='*70}")

## Step 8: Download Results

Download your validated_output.json file before disconnecting!

In [ ]:
from google.colab import files
import os

output_file = "validated_output.json"

if os.path.exists(output_file):
    print(f"[*] Downloading {output_file}...")
    files.download(output_file) 
    print(f"[OK] Download complete!")
    
    # Show file size
    file_size = os.path.getsize(output_file) / 1024  # KB
    print(f"[*] File size: {file_size:.2f} KB")
else:
    print(f"[ERROR] {output_file} not found")
    print(f"\nCurrent directory: {os.getcwd()}")
    print(f"Files in current directory:")
    !ls -la